# 🍎 CONTROL DE CALIDAD DE FRUTAS — VERSIÓN CORREGIDA v2
### Prof. Oswaldo Velez Lanngs, PhD
### EfficientNetV2-S · Grad-CAM · FastAPI Backend · Gradio Demo

---
**CAMBIOS CRÍTICOS vs versión anterior:**
- ✅ Preprocesamiento unificado (`preprocess_input` en TODOS los bloques)
- ✅ Label smoothing consistente en ambas fases de entrenamiento
- ✅ Cabeza mejorada: BatchNorm + Dense(512) + doble Dropout
- ✅ Data Augmentation extendido: saturation, hue, zoom, rotación
- ✅ Fine-tuning más conservador: últimas 50 capas, LR=2e-5
- ✅ Class weights calculados y aplicados correctamente
- ✅ Grad-CAM corregido (compatible con modelo anidado)
- ✅ Estrategia completa: 20 épocas HEAD → 40 épocas FINE-TUNE

## 📦 BLOQUE 1/13 — Instalación de Dependencias

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 1/13 — INSTALACIÓN DE DEPENDENCIAS                  ║
# ╚══════════════════════════════════════════════════════════════╝

print('🧹 Limpiando y configurando entorno...')
!pip uninstall -y gradio gradio_client huggingface_hub -q
!pip install -q huggingface_hub==0.24.5
!pip install -q 'gradio==4.44.1'
!pip install -q kaggle

print('✅ Dependencias instaladas. IMPORTANTE: Reinicia la sesión antes de continuar.')

## ⚙️ BLOQUE 2/13 — Importaciones y Configuración Global

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 2/13 — IMPORTACIONES Y CONFIGURACIÓN GLOBAL         ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys, json, hashlib, shutil, time, random, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image, ImageFile
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetV2S
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── GPU: memoria dinámica ─────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('=' * 55)
print(f'  TensorFlow  : {tf.__version__}')
print(f'  GPU activa  : {[g.name for g in gpus] or "CPU (sin GPU)"}')
try:
    ram_info = os.popen('free -h | grep Mem').read().split()
    print(f'  RAM total   : {ram_info[1]}  |  Libre: {ram_info[3]}')
except:
    pass
print('=' * 55)
print('✅ Configuración global lista')

## 📁 BLOQUE 3/13 — Montaje de Drive y Rutas del Proyecto

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 3/13 — MONTAJE DRIVE Y RUTAS                        ║
# ╚══════════════════════════════════════════════════════════════╝

import os
from google.colab import drive
drive.mount('/content/drive')

DATASET_ZIP   = '/content/drive/MyDrive/Dataset Detector Frutas/food-freshness-dataset.zip'
LOCAL_EXTRACT = '/content/dataset_local'

if not os.path.exists(LOCAL_EXTRACT):
    print('⏳ Descomprimiendo dataset...')
    !unzip -q "{DATASET_ZIP}" -d {LOCAL_EXTRACT}

# Buscar carpeta raíz con Fresh y Rotten
DATASET_ROOT = LOCAL_EXTRACT
for root, dirs, files in os.walk(LOCAL_EXTRACT):
    if 'Fresh' in dirs and 'Rotten' in dirs:
        DATASET_ROOT = root
        break

PROJECT_ROOT = '/content/drive/MyDrive/FrutasProyecto'
os.makedirs(PROJECT_ROOT, exist_ok=True)

# ── Hiperparámetros globales ──────────────────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 32
HEAD_EPOCHS   = 20    # ← AUMENTADO de 15 a 20
FINE_EPOCHS   = 40    # ← AUMENTADO para convergencia real
MAX_PER_CLASS = 2000
MIN_PER_CLASS = 100
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15

DIRS = {
    'checkpoints' : f'{PROJECT_ROOT}/checkpoints',
    'logs'        : f'{PROJECT_ROOT}/logs',
    'results'     : f'{PROJECT_ROOT}/results',
    'models'      : f'{PROJECT_ROOT}/models',
    'gradcam'     : f'{PROJECT_ROOT}/gradcam',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

STATE_FILE  = f'{PROJECT_ROOT}/training_state.json'
SPLIT_FILE  = f'{PROJECT_ROOT}/split_info.json'
BEST_CKPT   = f'{DIRS["checkpoints"]}/best_model.keras'
LATEST_CKPT = f'{DIRS["checkpoints"]}/latest_model.keras'

print(f'\nDataset Raíz: {DATASET_ROOT}')
if os.path.exists(DATASET_ROOT):
    subdirs = [d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))]
    print(f'   Carpetas principales: {subdirs}')
    n_files = sum(len(files) for _, _, files in os.walk(DATASET_ROOT))
    print(f'   Total archivos      : ~{n_files}')
print('✅ Rutas configuradas')

## 🔍 BLOQUE 4/13 — EDA: Análisis de Estructura y Clases

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 4/13 — EDA: ESCANEANDO FRESH Y ROTTEN               ║
# ╚══════════════════════════════════════════════════════════════╝

def scan_dataset_v2(root):
    root = Path(root)
    class_files = defaultdict(list)
    VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
    for img_path in root.rglob('*'):
        if img_path.suffix.lower() in VALID_EXT:
            parts = img_path.parts
            if len(parts) >= 2:
                class_name = f'{img_path.parent.parent.name}_{img_path.parent.name}'
                class_files[class_name].append(str(img_path))
    return dict(class_files)

print('🔍 Escaneando dataset...')
class_files  = scan_dataset_v2(DATASET_ROOT)
class_counts = {cls: len(files) for cls, files in class_files.items()}
total_imgs   = sum(class_counts.values())
CLASS_NAMES  = sorted(class_files.keys())
N_CLASSES    = len(CLASS_NAMES)

print('═' * 75)
print(f"  {'CLASE':<30} {'IMÁGENES':>10}  {'%':>7}  BARRA")
print('═' * 75)
max_count = max(class_counts.values())
for cls in CLASS_NAMES:
    cnt = class_counts[cls]
    pct = 100 * cnt / total_imgs
    bar = '█' * int(cnt / max_count * 20)
    print(f'  {cls:<30} {cnt:>10}  {pct:>6.1f}%  {bar}')
print('═' * 75)
print(f"  {'TOTAL':<30} {total_imgs:>10}")

counts = list(class_counts.values())
mn, mx = min(counts), max(counts)
ratio  = mx / mn if mn > 0 else float('inf')
print(f'\n  Ratio max/min : {ratio:.2f}x')
print(f'  Media/clase   : {np.mean(counts):.0f}')
print(f'  N° clases     : {N_CLASSES}')
print(f'\n  Clases detectadas: {CLASS_NAMES}')

## 📊 BLOQUE 5/13 — EDA: Visualizaciones y Muestras

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 5/13 — EDA: VISUALIZACIONES Y MUESTRAS              ║
# ╚══════════════════════════════════════════════════════════════╝

sorted_pairs             = sorted(class_counts.items(), key=lambda x: -x[1])
cls_sorted, cnt_sorted   = zip(*sorted_pairs)
palette                  = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(cls_sorted)))

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, N_CLASSES * 0.5 + 2)))
fig.suptitle('EDA — Distribución de Clases', fontsize=15, fontweight='bold')

bars = axes[0].barh(cls_sorted, cnt_sorted, color=palette, edgecolor='white', linewidth=0.5)
axes[0].axvline(x=np.mean(cnt_sorted), color='crimson', linestyle='--', lw=2,
                label=f'Media: {np.mean(cnt_sorted):.0f}')
for bar, cnt in zip(bars, cnt_sorted):
    axes[0].text(cnt + max(cnt_sorted)*0.01, bar.get_y() + bar.get_height()/2,
                 str(cnt), va='center', fontsize=8)
axes[0].set_xlabel('Número de imágenes', fontsize=11)
axes[0].set_title('Imágenes por clase', fontsize=12)
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

wedges, texts, autotexts = axes[1].pie(
    cnt_sorted, labels=cls_sorted, autopct='%1.1f%%',
    colors=palette, startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(7)
axes[1].set_title('Proporción de clases', fontsize=12)

plt.tight_layout()
plt.savefig(f"{DIRS['results']}/01_class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

# Muestras por clase
n_cols    = min(5, N_CLASSES)
n_rows    = (N_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.2))
fig.suptitle('EDA — Muestra de Imagen por Clase', fontsize=14, fontweight='bold')
axes_flat = np.array(axes).flatten() if N_CLASSES > 1 else [axes]

for i, cls in enumerate(CLASS_NAMES):
    sample_path = random.choice(class_files[cls])
    try:
        img = Image.open(sample_path).convert('RGB').resize((220, 220))
        axes_flat[i].imshow(img)
        status = '🟢' if 'fresh' in cls.lower() else '🔴'
        axes_flat[i].set_title(f'{status} {cls}\n({class_counts[cls]} imgs)', fontsize=8)
    except:
        axes_flat[i].set_title(f'{cls}\nERROR', fontsize=7, color='red')
    axes_flat[i].axis('off')

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.savefig(f"{DIRS['results']}/02_class_samples.png", dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA completado')

## 🧹 BLOQUE 6/13 — Duplicados, Corruptas y Calidad del Dataset

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 6/13 — DETECCIÓN DE DUPLICADOS E IMÁGENES CORRUPTAS ║
# ╚══════════════════════════════════════════════════════════════╝

def file_md5(path, chunk=8192):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.convert('RGB')
        return True
    except:
        return False

print('🔍 Verificando integridad y buscando duplicados...')

hash_map    = defaultdict(list)
corrupts    = []
total_files = sum(len(v) for v in class_files.values())
checked     = 0

for cls, paths in class_files.items():
    for p in paths:
        checked += 1
        if checked % 200 == 0:
            print(f'   Progreso: {checked}/{total_files} ({100*checked/total_files:.1f}%)', end='\r')
        if not is_valid_image(p):
            corrupts.append(p)
        else:
            h = file_md5(p)
            hash_map[h].append(p)

duplicates  = {h: ps for h, ps in hash_map.items() if len(ps) > 1}
n_dup_extra = sum(len(ps) - 1 for ps in duplicates.values())
valid_total = total_files - len(corrupts)

print(f'\n\n{"═"*55}')
print(f'  REPORTE DE CALIDAD DEL DATASET')
print(f'{"═"*55}')
print(f'  Total archivos    : {total_files:>8}')
print(f'  Imágenes válidas  : {valid_total:>8}')
print(f'  Corruptas         : {len(corrupts):>8}')
print(f'  Grupos duplicados : {len(duplicates):>8}')
print(f'  Extras duplicados : {n_dup_extra:>8}')
print(f'{"═"*55}')
print('✅ Análisis de calidad completado')

## ⚖️ BLOQUE 7/13 — Limpieza, Balanceo y Split Train/Val/Test

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 7/13 — LIMPIEZA, BALANCEO Y SPLIT                   ║
# ╚══════════════════════════════════════════════════════════════╝

corrupt_set    = set(corrupts)
dups_to_remove = set()
for h, paths in duplicates.items():
    for p in paths[1:]:
        dups_to_remove.add(p)

clean_class_files = defaultdict(list)
for cls, paths in class_files.items():
    for p in paths:
        if p not in corrupt_set and p not in dups_to_remove:
            clean_class_files[cls].append(p)

print('🧹 Dataset limpio:')
for cls in CLASS_NAMES:
    orig  = len(class_files[cls])
    clean = len(clean_class_files[cls])
    print(f'   {cls:<35} {clean:>6}  (eliminadas: {orig-clean})')

# Balanceo
clean_counts = {cls: len(files) for cls, files in clean_class_files.items()}
min_clean    = min(clean_counts.values())
target_count = min(MAX_PER_CLASS, min_clean * 3, max(clean_counts.values()))
target_count = max(target_count, MIN_PER_CLASS)
print(f'\n⚖️  Target por clase: {target_count}')

balanced = {}
for cls in CLASS_NAMES:
    files = clean_class_files[cls].copy()
    if len(files) > target_count:
        random.seed(SEED)
        files = random.sample(files, target_count)
    balanced[cls] = files

total_balanced = sum(len(v) for v in balanced.values())

# Split estratificado
print('\n✂️  Dividiendo Train / Val / Test...')
split_paths  = {'train': [], 'val': [], 'test': []}
split_labels = {'train': [], 'val': [], 'test': []}

for cls_idx, cls in enumerate(CLASS_NAMES):
    files    = balanced[cls]
    train_val, test = train_test_split(files, test_size=TEST_SPLIT, random_state=SEED, shuffle=True)
    val_frac        = VAL_SPLIT / (1.0 - TEST_SPLIT)
    train, val      = train_test_split(train_val, test_size=val_frac, random_state=SEED, shuffle=True)
    for split, subset in [('train', train), ('val', val), ('test', test)]:
        split_paths[split].extend(subset)
        split_labels[split].extend([cls_idx] * len(subset))

print(f"\n   {'Split':<8} {'Imágenes':>9} {'%':>7}")
print('   ' + '─'*26)
for split in ['train', 'val', 'test']:
    n   = len(split_paths[split])
    pct = 100 * n / total_balanced
    print(f'   {split.upper():<8} {n:>9}  ({pct:.1f}%)')

# Guardar split
split_info = {
    'class_names'  : CLASS_NAMES,
    'n_classes'    : N_CLASSES,
    'target_count' : target_count,
    'seed'         : SEED,
    'img_size'     : IMG_SIZE,
    'splits': {
        s: [{'path': p, 'label': l}
            for p, l in zip(split_paths[s], split_labels[s])]
        for s in ['train', 'val', 'test']
    }
}
with open(SPLIT_FILE, 'w') as f:
    json.dump(split_info, f, indent=2)

print(f'\n💾 Split guardado: {SPLIT_FILE}')
print('✅ Preparación de datos completada')

## 🔄 BLOQUE 8/13 — Pipeline tf.data con Data Augmentation MEJORADO
> **CAMBIOS v2:** Augmentation extendido con saturation, hue, zoom, rotación aleatoria

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 8/13 — PIPELINE tf.data MEJORADO v2                 ║
# ║  CAMBIO: Augmentation extendido + preprocess_input unificado ║
# ╚══════════════════════════════════════════════════════════════╝

import math, json, os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_ROOT = '/content/drive/MyDrive/FrutasProyecto'
SPLIT_FILE   = f'{PROJECT_ROOT}/split_info.json'
IMG_SIZE     = 224
BATCH_SIZE   = 32
SEED         = 42

if 'split_paths' not in dir() or not split_paths['train']:
    print('♨  Cargando splits desde Drive...')
    with open(SPLIT_FILE) as f:
        si = json.load(f)
    CLASS_NAMES  = si['class_names']
    N_CLASSES    = si['n_classes']
    split_paths  = {s: [d['path']  for d in si['splits'][s]] for s in ['train','val','test']}
    split_labels = {s: [d['label'] for d in si['splits'][s]] for s in ['train','val','test']}
    print('✅ Splits cargados')

PARALLEL_CALLS = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
# AUGMENTATION MEJORADO v2
# Se agregan: saturación, hue, zoom aleatorio y rotación
# ─────────────────────────────────────────────────────────────────────────────
@tf.function
def augment_image_v2(img, label):
    # Geométrico
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)

    # Zoom aleatorio (recorte + resize)
    crop_frac = tf.random.uniform([], 0.85, 1.0)
    h = tf.cast(tf.shape(img)[0], tf.float32)
    w = tf.cast(tf.shape(img)[1], tf.float32)
    crop_h = tf.cast(h * crop_frac, tf.int32)
    crop_w = tf.cast(w * crop_frac, tf.int32)
    img = tf.image.random_crop(img, size=[crop_h, crop_w, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # Color
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    img = tf.image.random_saturation(img, lower=0.7, upper=1.3)  # ← NUEVO
    img = tf.image.random_hue(img, max_delta=0.05)               # ← NUEVO

    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

# ─────────────────────────────────────────────────────────────────────────────
# PREPROCESAMIENTO UNIFICADO
# CRÍTICO: preprocess_input aquí y en TODOS los bloques de inferencia
# ─────────────────────────────────────────────────────────────────────────────
@tf.function
def load_image_and_label(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    label_oh = tf.one_hot(label, N_CLASSES)
    return img, label_oh

@tf.function
def apply_efficientnet_preprocessing(img, label):
    # SIEMPRE usar preprocess_input — nunca dividir por 255 manualmente
    img = tf.keras.applications.efficientnet_v2.preprocess_input(img)
    return img, label

def make_tf_dataset(paths, labels, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image_and_label, num_parallel_calls=PARALLEL_CALLS)
    if augment:
        ds = ds.map(augment_image_v2, num_parallel_calls=PARALLEL_CALLS)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.map(apply_efficientnet_preprocessing, num_parallel_calls=PARALLEL_CALLS)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print('🔧 Creando pipelines tf.data v2...')
train_ds = make_tf_dataset(split_paths['train'], split_labels['train'], augment=True, shuffle=True)
val_ds   = make_tf_dataset(split_paths['val'],   split_labels['val'])
test_ds  = make_tf_dataset(split_paths['test'],  split_labels['test'])

n_train = len(split_paths['train'])
n_val   = len(split_paths['val'])
n_test  = len(split_paths['test'])
print(f'\n✅ Pipelines listos:')
print(f'   Train : {n_train:>6} imgs → {math.ceil(n_train/BATCH_SIZE)} batches')
print(f'   Val   : {n_val:>6} imgs → {math.ceil(n_val/BATCH_SIZE)} batches')
print(f'   Test  : {n_test:>6} imgs → {math.ceil(n_test/BATCH_SIZE)} batches')

# Verificar que preprocess_input funciona (rango esperado: aprox -1 a 1)
for sample_imgs, sample_labels in train_ds.take(1):
    print(f'\n   Rango de píxeles tras preprocess_input: [{sample_imgs.numpy().min():.2f}, {sample_imgs.numpy().max():.2f}]')
    print(f'   (Rango correcto para EfficientNetV2: aprox [-1, 1])')

## 🏗️ BLOQUE 9/13 — Arquitectura del Modelo MEJORADA
> **CAMBIOS v2:** Cabeza mejorada (BatchNorm + Dense 512 + doble Dropout) · Class weights correctos

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 9/13 — ARQUITECTURA MEJORADA v2                     ║
# ║  CAMBIOS: Cabeza con BatchNorm+Dense(512) · class_weights    ║
# ╚══════════════════════════════════════════════════════════════╝

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print('🏗️ Construyendo arquitectura mejorada v2...')

# ── 1. Base preentrenada ───────────────────────────────────────────────────
base_model = keras.applications.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # Congelada para fase HEAD

# ── 2. Cabeza mejorada ─────────────────────────────────────────────────────
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
x = layers.BatchNormalization(name='bn_top')(x)           # ← NUEVO: estabiliza entrenamiento
x = layers.Dropout(0.4, name='dropout_1')(x)
x = layers.Dense(512, activation='relu', name='dense_512',
                 kernel_regularizer=keras.regularizers.l2(1e-4))(x)  # ← NUEVO
x = layers.BatchNormalization(name='bn_dense')(x)         # ← NUEVO
x = layers.Dropout(0.3, name='dropout_2')(x)              # ← NUEVO
outputs = layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)

model = keras.Model(inputs, outputs, name='FruitQuality_v2')

# ── 3. Compilación FASE 1 (HEAD) ──────────────────────────────────────────
# CRÍTICO: label_smoothing=0.1 se mantiene en AMBAS fases
LABEL_SMOOTHING = 0.1

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

print(model.summary())

total_params     = model.count_params()
trainable_params = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'\n   Parámetros totales     : {total_params:,}')
print(f'   Parámetros entrenables : {trainable_params:,}')

# ── 4. Class Weights ──────────────────────────────────────────────────────
# CRÍTICO: calcular y aplicar pesos para clases desbalanceadas
all_train_labels = split_labels['train']
unique_classes   = np.unique(all_train_labels)
cw_array         = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=all_train_labels
)
class_weights = dict(enumerate(cw_array))

print(f'\n⚖️  Class weights calculados ({len(class_weights)} clases):')
for idx, w in class_weights.items():
    print(f'   [{idx:02d}] {CLASS_NAMES[idx]:<35} → peso: {w:.4f}')

print(f'\n✅ Modelo listo. Label smoothing={LABEL_SMOOTHING}. Class weights listos.')

## 🚀 BLOQUE 10A/13 — FASE 1: Entrenamiento HEAD (Base Congelada)
> Ejecutar **una sola vez**. Entrena solo la cabeza. Target: val_accuracy > 60-70%

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 10A — FASE 1: HEAD TRAINING (Base Congelada)        ║
# ║  Ejecutar UNA VEZ antes del fine-tuning                     ║
# ╚══════════════════════════════════════════════════════════════╝

import os, math
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# Rutas de checkpoint
CHECKPOINT_PATH_HEAD = f"{DIRS['checkpoints']}/head_best.keras"

# Asegurar que la base esté congelada
base_model = model.layers[1]
base_model.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

callbacks_head = [
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH_HEAD,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.TensorBoard(
        log_dir=f"{DIRS['logs']}/head",
        histogram_freq=1
    )
]

print('🔒 FASE 1: HEAD TRAINING')
print(f'   Épocas máx : {HEAD_EPOCHS}')
print(f'   LR inicial : 1e-3')
print(f'   Base       : CONGELADA')
print(f'   Label smooth: {LABEL_SMOOTHING}')
print()

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks_head,
    class_weight=class_weights
)

# Graficar curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('FASE 1 — HEAD Training', fontsize=14)

axes[0].plot(history_head.history['accuracy'],     label='Train Acc')
axes[0].plot(history_head.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_head.history['loss'],     label='Train Loss')
axes[1].plot(history_head.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DIRS['results']}/head_training_curves.png", dpi=150)
plt.show()

best_head_acc = max(history_head.history['val_accuracy'])
print(f'\n✅ FASE 1 completada. Mejor val_accuracy: {best_head_acc*100:.2f}%')
print(f'💾 Mejor modelo guardado en: {CHECKPOINT_PATH_HEAD}')

## 🔓 BLOQUE 10B/13 — FASE 2: Fine-Tuning (Últimas 50 capas)
> Ejecutar **después del BLOQUE 10A**. Requiere val_accuracy > 50% al terminar fase 1.
> LR muy bajo (2e-5) para no destruir pesos de ImageNet

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 10B — FASE 2: FINE-TUNING (Últimas 50 capas)        ║
# ║  CAMBIOS v2: LR=2e-5, últimas 50 (no 100), label_smooth     ║
# ╚══════════════════════════════════════════════════════════════╝

import os, json
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# Cargar mejor modelo de fase HEAD si el Kernel fue reiniciado
CHECKPOINT_PATH_HEAD = f"{DIRS['checkpoints']}/head_best.keras"
CHECKPOINT_PATH_FINE = f"{DIRS['checkpoints']}/finetune_best.keras"

if 'model' not in dir() or model is None:
    print('♨  Cargando modelo HEAD desde Drive...')
    model = keras.models.load_model(CHECKPOINT_PATH_HEAD)
    print('✅ Modelo cargado')

if 'class_weights' not in dir() or class_weights is None:
    print('♨  Recalculando class_weights...')
    from sklearn.utils.class_weight import compute_class_weight
    cw_array     = compute_class_weight('balanced',
                                        classes=np.unique(split_labels['train']),
                                        y=split_labels['train'])
    class_weights = dict(enumerate(cw_array))

# ── Descongelar últimas 50 capas del EfficientNet ────────────────────────────
# CAMBIO v2: 50 capas en lugar de 100 — más conservador, menos riesgo de degradar
print('🔓 Descongelando últimas 50 capas del backbone...')
base_model = None
for layer in model.layers:
    if 'efficientnet' in layer.name.lower():
        base_model = layer
        break

base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 50  # Congelar todo menos las últimas 50

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

# Contar capas entrenables
trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count    = sum(1 for l in base_model.layers if not l.trainable)
print(f'   Capas EfficientNet entrenables : {trainable_count}')
print(f'   Capas EfficientNet congeladas  : {frozen_count}')

# ── Recompilar con LR muy bajo ────────────────────────────────────────────────
# CAMBIO v2: 2e-5 en lugar de 1e-5 — converge más rápido sin destruir pesos
# CRÍTICO: Mantener label_smoothing=0.1 igual que en fase HEAD
FINE_LR = 2e-5
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_LR),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

callbacks_fine = [
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH_FINE,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,           # Más paciencia en fine-tuning
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.3,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.TensorBoard(
        log_dir=f"{DIRS['logs']}/finetune",
        histogram_freq=1
    )
]

print(f'\n🔓 FASE 2: FINE-TUNING')
print(f'   Épocas máx    : {FINE_EPOCHS}')
print(f'   LR inicial    : {FINE_LR}')
print(f'   Capas activas : últimas 50 del backbone')
print(f'   Label smooth  : {LABEL_SMOOTHING}')
print()

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_EPOCHS,
    callbacks=callbacks_fine,
    class_weight=class_weights
)

# Graficar curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('FASE 2 — Fine-Tuning', fontsize=14)

axes[0].plot(history_fine.history['accuracy'],     label='Train Acc')
axes[0].plot(history_fine.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_fine.history['loss'],     label='Train Loss')
axes[1].plot(history_fine.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DIRS['results']}/finetune_training_curves.png", dpi=150)
plt.show()

best_fine_acc = max(history_fine.history['val_accuracy'])
print(f'\n✅ FASE 2 completada. Mejor val_accuracy: {best_fine_acc*100:.2f}%')
print(f'💾 Mejor modelo guardado en: {CHECKPOINT_PATH_FINE}')

## 📊 BLOQUE 11/13 — Evaluación Completa del Modelo

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 11/13 — EVALUACIÓN COMPLETA                         ║
# ╚══════════════════════════════════════════════════════════════╝

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras

print('📊 EVALUACIÓN DEL MODELO...')

# Cargar mejor modelo si no está en memoria
CHECKPOINT_PATH_FINE = f"{DIRS['checkpoints']}/finetune_best.keras"
if 'model' not in dir() or model is None:
    model = keras.models.load_model(CHECKPOINT_PATH_FINE)
    print('✅ Modelo cargado desde Drive')

# Evaluar en test set
loss, accuracy = model.evaluate(test_ds, verbose=1)
print(f'\n  ✅ Test Accuracy : {accuracy * 100:.2f}%')
print(f'  📉 Test Loss     : {loss:.4f}')

# Predicciones detalladas
y_true = []
y_pred_probs = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_pred = np.argmax(y_pred_probs, axis=1)

print('\n📄 REPORTE DE CLASIFICACIÓN:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Matriz de Confusión — Clasificación de Calidad de Frutas', fontsize=16, pad=20)
plt.xlabel('Predicción', fontsize=14)
plt.ylabel('Real', fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f"{DIRS['results']}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# Análisis de errores por clase
print('\n🔎 Precisión por clase:')
for i, cls in enumerate(CLASS_NAMES):
    mask    = np.array(y_true) == i
    correct = np.sum(np.array(y_pred)[mask] == i)
    total   = np.sum(mask)
    acc_cls = correct / total if total > 0 else 0
    bar     = '█' * int(acc_cls * 20)
    status  = '✅' if acc_cls >= 0.85 else ('⚠️' if acc_cls >= 0.7 else '❌')
    print(f'  {status} {cls:<35} {acc_cls*100:5.1f}%  {bar}')

## 🔥 BLOQUE 12/13 — Grad-CAM CORREGIDO
> **CORRECCIÓN CRÍTICA:** Compatible con modelo anidado EfficientNet · Preprocesamiento unificado

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 12/13 — GRAD-CAM CORREGIDO v2                       ║
# ║  FIX: Preprocesamiento unificado + modelo anidado            ║
# ╚══════════════════════════════════════════════════════════════╝

import os, cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

os.makedirs(DIRS['gradcam'], exist_ok=True)

# ── Extraer sub-modelo para Grad-CAM ─────────────────────────────────────────
def build_gradcam_model(model):
    """Construye sub-modelo Grad-CAM compatible con EfficientNet anidado."""
    base_model = None
    for layer in model.layers:
        if 'efficientnet' in layer.name.lower():
            base_model = layer
            break
    if base_model is None:
        base_model = model.layers[1]

    # Encontrar última capa conv
    last_conv_name = None
    for layer in reversed(base_model.layers):
        if 'conv' in layer.name.lower():
            last_conv_name = layer.name
            break

    print(f'  🎯 Modelo base     : {base_model.name}')
    print(f'  🎯 Última capa conv: {last_conv_name}')

    grad_submodel = tf.keras.Model(
        inputs=base_model.input,
        outputs=[base_model.get_layer(last_conv_name).output, base_model.output]
    )
    return base_model, grad_submodel


def compute_gradcam(model, base_model, grad_submodel, img_array_preprocessed, class_idx):
    """Calcula heatmap Grad-CAM. img_array_preprocessed ya tiene preprocess_input aplicado."""
    img_tensor = tf.cast(tf.expand_dims(img_array_preprocessed, axis=0), tf.float32)

    with tf.GradientTape() as tape:
        # Pasar por capas previas al EfficientNet (ignorar InputLayer)
        x = img_tensor
        for layer in model.layers:
            if layer == base_model:
                break
            if 'InputLayer' not in layer.__class__.__name__:
                x = layer(x)

        conv_outputs, base_outputs = grad_submodel(x)
        tape.watch(conv_outputs)

        # Pasar por la cabeza clasificadora
        base_idx = model.layers.index(base_model)
        x_top = base_outputs
        for layer in model.layers[base_idx + 1:]:
            x_top = layer(x_top)

        loss = x_top[:, class_idx]

    grads          = tape.gradient(loss, conv_outputs)
    guided_grads   = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap        = conv_outputs[0] @ guided_grads[..., tf.newaxis]
    heatmap        = tf.squeeze(heatmap)
    heatmap        = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


def overlay_gradcam(original_rgb_float, heatmap, alpha=0.45):
    """Superpone heatmap sobre imagen original. Ambas en [0,1]."""
    heatmap_resized = cv2.resize(heatmap, (original_rgb_float.shape[1], original_rgb_float.shape[0]))
    heatmap_colored = plt.cm.jet(heatmap_resized)[..., :3]
    superimposed    = heatmap_colored * alpha + original_rgb_float * (1 - alpha)
    return np.clip(superimposed, 0, 1)


# ── FUNCIÓN DE INFERENCIA CORREGIDA ──────────────────────────────────────────
# CORRECCIÓN CRÍTICA: preprocess_input en lugar de /255
def predict_and_visualize(image_path_or_array, model, show=True):
    """
    Realiza predicción y genera Grad-CAM.
    Usa preprocess_input — mismo que el pipeline de entrenamiento.
    """
    # Cargar imagen
    if isinstance(image_path_or_array, str):
        img_pil = Image.open(image_path_or_array).convert('RGB')
    else:
        # numpy array (ej: desde Gradio con type='numpy')
        img_pil = Image.fromarray(image_path_or_array.astype(np.uint8)).convert('RGB')

    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np      = np.array(img_resized, dtype=np.float32)  # [0, 255]

    # ─ CORRECCIÓN CRÍTICA ────────────────────────────────────────────────────
    # Preprocesar IGUAL que en entrenamiento: preprocess_input
    # NUNCA hacer img_np / 255.0 aquí
    img_preprocessed = tf.keras.applications.efficientnet_v2.preprocess_input(
        img_np.copy()
    )
    # ─────────────────────────────────────────────────────────────────────────

    # Predicción
    inp          = tf.expand_dims(img_preprocessed, axis=0)
    preds        = model.predict(inp, verbose=0)[0]
    clase_idx    = int(np.argmax(preds))
    confianza    = float(preds[clase_idx]) * 100
    nombre_clase = CLASS_NAMES[clase_idx]

    # Grad-CAM
    base_model_local, grad_submodel_local = build_gradcam_model(model)
    heatmap   = compute_gradcam(model, base_model_local, grad_submodel_local,
                                 img_preprocessed, clase_idx)
    img_norm  = img_np / 255.0  # Solo para overlay visual — no para predicción
    overlay   = overlay_gradcam(img_norm, heatmap)

    if show:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        es_fresca = 'Fresh' in nombre_clase
        color     = 'darkgreen' if es_fresca else 'red'
        estado    = '🟢 FRESCA' if es_fresca else '🔴 PODRIDA'

        axes[0].imshow(img_np.astype(np.uint8))
        axes[0].set_title('Original', fontsize=12)
        axes[0].axis('off')

        axes[1].imshow(plt.cm.jet(cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))))
        axes[1].set_title('Heatmap Grad-CAM', fontsize=12)
        axes[1].axis('off')

        axes[2].imshow(overlay)
        axes[2].set_title(f'{estado}\n{nombre_clase}\nConfianza: {confianza:.1f}%',
                          fontsize=11, color=color)
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

    return nombre_clase, confianza, overlay


# ── TEST: probar con imágenes del test set ────────────────────────────────────
print('🔥 Grad-CAM — Probando con 6 imágenes del test set...')
sample_test = random.sample(split_paths['test'], min(6, len(split_paths['test'])))

for path in sample_test:
    nombre, conf, _ = predict_and_visualize(path, model)
    print(f'  → {nombre:<35} {conf:.1f}%')

## 🌐 BLOQUE 13/13 — Interfaz Gradio con Preprocesamiento Correcto

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOQUE 13/13 — GRADIO v2 CON PREPROCESAMIENTO CORRECTO     ║
# ║  FIX CRÍTICO: preprocess_input en lugar de /255             ║
# ╚══════════════════════════════════════════════════════════════╝

import gradio as gr
import time

def predict_fruit_web(image):
    """Recibe imagen PIL desde Gradio. Usa preprocess_input unificado."""
    if image is None:
        return {}, None, '⚠️ Sube una imagen primero.'

    img_r    = image.resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
    img_np   = np.array(img_r, dtype=np.float32)

    # ─ CORRECCIÓN CRÍTICA ────────────────────────────────────────────────────
    inp_np   = tf.keras.applications.efficientnet_v2.preprocess_input(img_np.copy())
    # ─────────────────────────────────────────────────────────────────────────

    inp_t    = tf.expand_dims(inp_np, 0)

    t0       = time.time()
    probs    = model(inp_t, training=False).numpy()[0]
    ms       = (time.time() - t0) * 1000

    pred_idx  = int(np.argmax(probs))
    pred_name = CLASS_NAMES[pred_idx]
    conf      = float(probs[pred_idx])
    es_fresca = 'Fresh' in pred_name
    quality   = '✅ FRUTA FRESCA' if es_fresca else '❌ FRUTA DAÑADA'

    # Grad-CAM solo para Rotten
    overlay = img_np.astype(np.uint8)
    if not es_fresca:
        try:
            base_m, grad_sm = build_gradcam_model(model)
            hm      = compute_gradcam(model, base_m, grad_sm, inp_np, pred_idx)
            img_01  = img_np / 255.0
            ov      = overlay_gradcam(img_01, hm)
            overlay = (ov * 255).astype(np.uint8)
        except Exception as e:
            print(f'Grad-CAM warning: {e}')

    prob_dict = {CLASS_NAMES[i]: float(probs[i]) for i in range(N_CLASSES)}
    info_md   = (
        f'### {quality}\n\n'
        f'| Campo | Valor |\n|---|---|\n'
        f'| **Clase** | `{pred_name}` |\n'
        f'| **Confianza** | `{conf:.1%}` |\n'
        f'| **Tiempo** | `{ms:.1f} ms` |\n'
        f'| **Modelo** | EfficientNetV2-S v2 |\n'
    )

    return prob_dict, Image.fromarray(overlay), info_md


# Interfaz
sample_paths = random.sample(split_paths['test'], min(8, len(split_paths['test'])))

with gr.Blocks(title='Control de Calidad de Frutas v2', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🍎 Control de Calidad de Frutas — v2
    **Modelo:** EfficientNetV2-S v2 · **Framework:** TensorFlow/Keras
    """)
    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type='pil', label='📷 Imagen de la Fruta',
                              sources=['upload', 'webcam', 'clipboard'], height=320)
            btn    = gr.Button('🔍 Analizar Fruta', variant='primary', size='lg')
        with gr.Column(scale=1):
            cam_out = gr.Image(label='🔥 Grad-CAM (solo frutas dañadas)', height=320)
    with gr.Row():
        label_out = gr.Label(label='📊 Probabilidades', num_top_classes=min(N_CLASSES, 12))
        info_out  = gr.Markdown()
    gr.Examples(examples=[[p] for p in sample_paths], inputs=img_in,
                label='🖼️ Ejemplos del Test Set')
    btn.click(fn=predict_fruit_web, inputs=img_in, outputs=[label_out, cam_out, info_out])

print('🚀 Lanzando Gradio v2...')
demo.launch(share=True, debug=False, show_error=True, server_name='0.0.0.0')

## 📦 BLOQUE BONUS — Exportar modelo final para FastAPI Backend

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BONUS — EXPORTAR MODELO PARA FASTAPI BACKEND               ║
# ║  Guarda como .keras (recomendado) y .tflite (móvil)         ║
# ╚══════════════════════════════════════════════════════════════╝

import os, tensorflow as tf
from tensorflow import keras

CHECKPOINT_PATH_FINE = f"{DIRS['checkpoints']}/finetune_best.keras"
EXPORT_KERAS         = f"{DIRS['models']}/fruit_classifier.keras"
EXPORT_TFLITE        = f"{DIRS['models']}/fruit_quality_f16.tflite"

print('📦 Exportando modelo...')

# Cargar mejor modelo
if 'model' not in dir() or model is None:
    model = keras.models.load_model(CHECKPOINT_PATH_FINE)

# ── 1. Guardar .keras (para FastAPI) ─────────────────────────────────────────
model.save(EXPORT_KERAS)
size_keras = os.path.getsize(EXPORT_KERAS) / 1e6
print(f'✅ .keras guardado: {EXPORT_KERAS} ({size_keras:.1f} MB)')

# ── 2. Guardar .tflite (para móvil) ──────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations          = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open(EXPORT_TFLITE, 'wb') as f:
    f.write(tflite_model)
size_tflite = os.path.getsize(EXPORT_TFLITE) / 1e6
print(f'✅ .tflite guardado: {EXPORT_TFLITE} ({size_tflite:.1f} MB)')

print(f'\n📋 RESUMEN DE EXPORTACIÓN:')
print(f'  Backend FastAPI → descarga: {EXPORT_KERAS}')
print(f'  Móvil TFLite    → descarga: {EXPORT_TFLITE}')
print(f'\n  Coloca fruit_classifier.keras en: backend/model/fruit_classifier.keras')